## Three stages similarity calculation

In [1]:
import numpy as np
import pandas as pd


def load_mean_adj(graph_path, label_path):
    graphs = np.load(graph_path)
    adj = graphs.mean(axis=0)
    df = pd.read_csv(label_path)
    labels = df.iloc[:, 0].astype(str).str.strip().values
    return adj, labels


# paths
stage1_graph_path_2 = './result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy'
stage1_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage2_graph_path_2 = './result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy'
stage2_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage3_graph_path_2 = './result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5_seed42/dynamic_full_graphs.npy'
stage3_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'


A1, L1 = load_mean_adj(stage1_graph_path_2, stage1_label_path_2)
A2, L2 = load_mean_adj(stage2_graph_path_2, stage2_label_path_2)
A3, L3 = load_mean_adj(stage3_graph_path_2, stage3_label_path_2)


# find intersection regions
common = sorted(list(set(L1) & set(L2) & set(L3)))


def reorder(adj, labels, common):
    idx = [np.where(labels == c)[0][0] for c in common]
    return adj[np.ix_(idx, idx)]


A1c = reorder(A1, L1, common)
A2c = reorder(A2, L2, common)
A3c = reorder(A3, L3, common)


def edge_mae(A, B, eps=0, mode="all"):
    if A.shape != B.shape:
        raise ValueError("A and B must have the same shape.")

    n = A.shape[0]

    if mode == "offdiag":
        region_mask = ~np.eye(n, dtype=bool)
    elif mode == "diagonal":
        region_mask = np.eye(n, dtype=bool)
    elif mode == "all":
        region_mask = np.ones((n, n), dtype=bool)
    else:
        raise ValueError("mode must be 'offdiag', 'diagonal', or 'all'.")

    valid_mask = region_mask & (
        (np.abs(A) > eps) | (np.abs(B) > eps)
    )

    if valid_mask.sum() == 0:
        return np.nan

    return np.abs(A[valid_mask] - B[valid_mask]).mean()


mae12 = edge_mae(A1c, A2c)
mae13 = edge_mae(A1c, A3c)
mae23 = edge_mae(A2c, A3c)

print("Intersection brain regions:", len(common))
print("Edge MAE (Stage1 vs Stage2):", round(mae12, 4))
print("Edge MAE (Stage1 vs Stage3):", round(mae13, 4))
print("Edge MAE (Stage2 vs Stage3):", round(mae23, 4))

Intersection brain regions: 89
Edge MAE (Stage1 vs Stage2): 0.1353
Edge MAE (Stage1 vs Stage3): 0.1697
Edge MAE (Stage2 vs Stage3): 0.1737


In [2]:
stage_variation = np.mean([mae12, mae13, mae23])

print(f"Stage variation: {stage_variation:.4f}")

Stage variation: 0.1596


In [3]:
import numpy as np
import pandas as pd
import mpmath as mp
from scipy.stats import pearsonr

mp.mp.dps = 5000

# ============================================================
# load learned graphs
# ============================================================
def load_mean_adj(graph_path, label_path):
    graphs = np.load(graph_path)
    adj = graphs.mean(axis=0)

    df = pd.read_csv(label_path)
    labels = df.iloc[:, 0].astype(str).str.strip().values

    return adj, labels


# ============================================================
# reorder matrix
# ============================================================
def reorder(adj, labels, common):
    labels = np.asarray(labels).astype(str)
    idx = [np.where(labels == c)[0][0] for c in common]
    return adj[np.ix_(idx, idx)]


# ============================================================
# high-precision Pearson p-value
# ============================================================
def pearson_p_mpmath(r, n, digits=30):
    r = mp.mpf(str(abs(r)))
    n = mp.mpf(str(n))

    df = n - 2

    # two-sided p-value for Pearson correlation
    # p = I_{1-r^2}(df/2, 1/2)
    x = 1 - r**2
    p = mp.betainc(df / 2, mp.mpf("0.5"), 0, x, regularized=True)

    return mp.nstr(p, digits, min_fixed=0, max_fixed=0)


# ============================================================
# compare matrices
# ============================================================
def compare_matrix(A,B):
    A,B=np.asarray(A,float),np.asarray(B,float)
    m=~np.eye(A.shape[0],dtype=bool)
    a,b=np.abs(A[m]),np.abs(B[m])
    v=np.isfinite(a)&np.isfinite(b); a,b=a[v],b[v]
    r,_=pearsonr(a,b)
    return {
        "MAE":np.mean(np.abs(a-b)),
        "RMSE":np.sqrt(np.mean((a-b)**2)),
        "Pearson_r":r,
        "Pearson_p":pearson_p_mpmath(r,len(a),digits=3)
    }

# ============================================================
# learned graphs
# ============================================================
stage1_graph_path_2 = './result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy'
stage1_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage2_graph_path_2 = './result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy'
stage2_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage3_graph_path_2 = './result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5_seed42/dynamic_full_graphs.npy'
stage3_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'



# ============================================================
# load learned graphs
# ============================================================
A1, L1 = load_mean_adj(stage1_graph_path_2, stage1_label_path_2)
A2, L2 = load_mean_adj(stage2_graph_path_2, stage2_label_path_2)
A3, L3 = load_mean_adj(stage3_graph_path_2, stage3_label_path_2)


# ============================================================
# load layer matrix
# ============================================================
df_layers = pd.read_excel(
    '../../SEU_net_wo_corticalLayers.xlsx',
    header=1,
    index_col=1
)

df_layers = df_layers.iloc[:, 1:]

layer_labels = df_layers.index.astype(str).str.strip().values
A_layer = df_layers.values.astype(float)


# ============================================================
# intersection
# ============================================================
common = sorted(list(set(L1) & set(L2) & set(L3) & set(layer_labels)))

print(f"\nIntersection brain regions: {len(common)}")


# ============================================================
# reorder
# ============================================================
A1c = reorder(A1, L1, common)
A2c = reorder(A2, L2, common)
A3c = reorder(A3, L3, common)
Alc = reorder(A_layer, layer_labels, common)


# ============================================================
# compare
# ============================================================
results = {
    "Stage1_vs_Layer": compare_matrix(A1c, Alc),
    "Stage2_vs_Layer": compare_matrix(A2c, Alc),
    "Stage3_vs_Layer": compare_matrix(A3c, Alc),
}

df_results = pd.DataFrame(results).T


# ============================================================
# print only
# ============================================================
df_print = df_results.copy()

df_print["MAE"] = df_print["MAE"].map(lambda x: f"{x:.4f}")
df_print["RMSE"] = df_print["RMSE"].map(lambda x: f"{x:.4f}")
df_print["Pearson_r"] = df_print["Pearson_r"].map(lambda x: f"{x:.4f}")

print("\nComparison results:")
print(df_print)


Intersection brain regions: 89

Comparison results:
                    MAE    RMSE Pearson_r Pearson_p
Stage1_vs_Layer  0.1632  0.7297    0.1188  5.25e-26
Stage2_vs_Layer  0.1635  0.7308    0.1115  4.45e-23
Stage3_vs_Layer  0.1635  0.7324    0.0977  4.62e-18


In [4]:
mean_pearson_r = df_results["Pearson_r"].astype(float).mean()

print(f"\nMean Pearson correlation: {mean_pearson_r:.4f}")


Mean Pearson correlation: 0.1093


In [5]:
import numpy as np, pandas as pd, mpmath as mp
from scipy.stats import pearsonr

mp.mp.dps = 100

paths = [
    ('./result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv'),
    ('./result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0_seed42/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv'),
    ('./result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5_seed42/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv')
]

def load(g,l):
    return np.load(g).mean(0), pd.read_csv(l).iloc[:,0].astype(str).str.strip().values

def pval(r,n):
    x=1-mp.mpf(str(abs(r)))**2
    return mp.nstr(mp.betainc((n-2)/2,.5,0,x,regularized=True),3,min_fixed=0,max_fixed=0)

A,L = zip(*[load(*p) for p in paths])
common = sorted(set(L[0]) & set(L[1]) & set(L[2]))

A = [
    a[np.ix_(
        [np.where(l==c)[0][0] for c in common],
        [np.where(l==c)[0][0] for c in common]
    )]
    for a,l in zip(A,L)
]

mask = ~np.eye(len(common),dtype=bool)
pairs = [(0,1),(0,2),(1,2)]
names = ["I–II","I–III","II–III"]

res = []
for (i,j),name in zip(pairs,names):
    x,y=A[i][mask],A[j][mask]
    m=np.isfinite(x)&np.isfinite(y)
    x,y=x[m],y[m]
    r,_=pearsonr(x,y)
    res.append([name,np.mean(abs(x-y)),np.sqrt(np.mean((x-y)**2)),r,pval(r,len(x))])

df = pd.DataFrame(res,columns=["Comparison","MAE","RMSE","Pearson_r","Pearson_p"])
print(df.to_string(index=False,formatters={
    "MAE":"{:.4f}".format,
    "RMSE":"{:.4f}".format,
    "Pearson_r":"{:.4f}".format
}))

r=df["Pearson_r"].to_numpy()
rel=(1-r)/(1-r).sum()

print("\nRelative stage separation:")
for n,x in zip(names,rel):
    print(f"{n}: {x:.1%}")

Comparison    MAE   RMSE Pearson_r  Pearson_p
      I–II 0.0147 0.1013    0.6837 4.54e-1074
     I–III 0.0184 0.1122    0.5919  9.27e-736
    II–III 0.0189 0.1124    0.6087  1.23e-789

Relative stage separation:
I–II: 28.3%
I–III: 36.6%
II–III: 35.1%
